In [ ]:
import os
import torch
import torch.nn as nn

In [ ]:
def train_loop(epochs, train_dl, val_dl, model, loss_fn, optimizer, scheduler, acc_fns, train_tfms, norm, val_tfms, patience=10):
    model.train()
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    
    os.makedirs('checkpoints', exist_ok=True)
    
    for epoch in range(epochs):
        train_loss = 0.0
        
        for batch_idx, batch in enumerate(train_dl):
            optimizer.zero_grad()
            
            mask = batch['mask'].unsqueeze(1).float() 
            combined = torch.cat([batch['image'], mask], dim=1)  
            
            augmented = train_tfms(combined) 
            
            X = augmented[:, :6, :, :] 
            y = augmented[:, 6:, :, :] 
            
            X = norm(X).to(device)
            y = remap_mask(y).type(torch.long).to(device)
            
            outputs = model(X) #Для FCN: outputs = model(X)['out']
            pred = outputs
            
            loss = loss_fn(pred, y)
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            
            train_loss += loss.item() / len(train_dl)
        
        model.eval()
        val_loss = 0.0
        accs = [0.0] * len(acc_fns)
        
        with torch.no_grad():
            for batch in val_dl:
                X = val_tfms(batch['image']).to(device)
                y = remap_mask(batch['mask']).type(torch.long).to(device)
                
                pred = model(X) #Для FCN: outputs = model(X)['out']
                
                loss = loss_fn(pred, y)
                val_loss += loss.item() / len(val_dl)
                
                for i, acc_fn in enumerate(acc_fns):
                    accs[i] += acc_fn(pred, y).item() / len(val_dl)
                    
        if scheduler is not None:
            scheduler.step(val_loss)
        print(f"Epoch {epoch+1}: Train={train_loss:.4f} Val={val_loss:.4f} "
              f"OA={accs[0]:.3f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
            
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': best_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
                'train_loss': train_loss
            }
            if scheduler is not None:
                checkpoint['scheduler_state_dict'] = scheduler.state_dict()
            
            torch.save(checkpoint, 'checkpoints/best_model.pth') # Сохранение чекпоинтов
            print(f"New best Focal: {best_val_loss:.4f} - Checkpoint saved")
                    
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping") # Работа EarlyStopping
                break
    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(best_state, 'best_model.pth')
        print(f"Final best model saved with val_loss: {best_val_loss:.4f}")
    
    return model

In [ ]:
optimizer = torch.optim.AdamW(pan_model.parameters(), lr=1e-4, weight_decay=1e-3) #Оптимизатор
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5) # Планировщик

model = train_loop(
    epochs=100,
    train_dl=train_dataloader,
    val_dl=val_dataloader,
    model=model, # Выбрать
    loss_fn=loss_fn, # Выбрать
    optimizer=optimizer,
    scheduler=scheduler,
    acc_fns=[oa],
    norm=norm,
    train_tfms=train_tfms,
    val_tfms=val_tfms,
    patience=8
    )